# 17 · "Filho da terra": redutos e bolsões

**Redutos**: Ronaldo Caiado (nº 55) em **Goiás** (ex-governador de GO) e Zema (nº 30) em **Minas Gerais** (governador de MG). Razão = % na UF ÷ % nacional (ambos sobre válidos).

**Bolsões**: candidatos com **< 5% nacional** (todos exceto Lula e Flávio). Um município é **qualificado** se seu % sobre válidos supera **3× a média nacional** do candidato (% sobre válidos BR, sem exterior). **Bolsão** = componente conexa (queen) de municípios qualificados.

**Ressalvas**: (1) bolsão é propriedade de municípios, não de eleitores; (2) para candidatos com pouca votação nacional, 3× é um limiar baixo em números absolutos e muitos bolsões são de 1–3 municípios pequenos (ruído); (3) a média nacional é ponderada (sobre válidos), não a média simples dos municípios.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path("..") / "src"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from eleicao.analise import composicao as cp, base, carga, graficos as g
from eleicao.analise import (
    participacao as pa,
    concentracao as cc,
    swing as sw,
    perfil,
    espacial as esp,
    bolsoes as bo,
    mapa_mente as mm,
)

## Redutos

In [ ]:
for uf_, nr in (("go", 55), ("mg", 30)):
    r = cp.tabela_reduto(uf_, nr)
    print(
        f"nº {nr} em {uf_.upper()}: {r['pct_uf_sobre_validos']:.2f}% dos válidos da UF (ponderado) | nacional {r['pct_nacional_sobre_validos']:.2f}% | razão {r['razao_vs_nacional']:.2f}x | municípios acima de 3x: {r['n_municipios_acima_3x']} de {r['n_municipios_uf']}"
    )

## Bolsões de todos os candidatos com < 5% nacional

In [ ]:
nrs = [70, 14, 55, 30, 80, 16, 27, 21, 35, 29]
B = cp.bolsoes_por_candidato(nrs, limiar_x=3.0)
resumo = cp.resumo_bolsoes(B)
resumo["nome"] = resumo.nr_candidato.map(g.NOME_CURTO)
print(resumo.round(3).to_string(index=False))

## Maiores bolsões (por nº de municípios) de cada candidato

In [ ]:
for nr in nrs:
    bl = B[str(nr)]["bolsoes"].sort_values("n_municipios", ascending=False).head(3)
    print(
        nr,
        g.NOME_CURTO.get(nr),
        f"(média nacional {B[str(nr)]['media_nacional_pct']:.3f}%, limiar {B[str(nr)]['limiar_pct']:.3f}%)",
    )
    if len(bl) == 0:
        print("   nenhum bolsão")
        continue
    print(
        bl[["n_municipios", "eleitorado", "municipio_maior_pct", "uf_maior_pct", "pct_maximo"]]
        .round(3)
        .to_string(index=False)
    )

## Mapa dos bolsões de Caiado (GO) e Zema (MG)

In [ ]:
b = cp.base_2026()
fig, ax = plt.subplots(1, 2, figsize=(12, 7))
for a, (nr, nome) in zip(ax, [("55", "Caiado (55)"), ("30", "Zema (30)")]):
    mp = cp.malha_bolsoes(B, nr, b)
    cor = mp["em_bolsao"].map({True: g.cor_candidato(int(nr)), False: "#ececec"})
    mp.plot(color=cor, linewidth=0, ax=a)
    a.set_axis_off()
    a.set_title(f"{nome}: municípios em bolsão (> 3x média nacional)", loc="left", fontsize=10)
g.salvar(fig, "bolsoes_caiado_zema_mapa")
plt.show()